# First Set Up the GPU in Kaggle 

In [1]:
!nvidia-smi # to check if gpu is loaded for current session

Wed Aug 26 08:19:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Download PreRequisits

In [2]:
%%writefile requirements.txt

--extra-index-url https://download.pytorch.org/whl/cu117
torch==2.5.1+cu117
torchvision==0.20.1+cu117
accelerate>=1.1.0
tqdm>=4.66.0
numpy>=1.24.3,<2.0.0

Writing requirements.txt


In [3]:
!pip list | grep -E "torch|torchvision|accelerate|numpy|tqdm"

accelerate                               1.13.0
numpy                                    2.0.2
pytorch-ignite                           0.5.4
pytorch-lightning                        2.6.5
torch                                    2.10.0+cu128
torchao                                  0.10.0
torchaudio                               2.10.0+cu128
torchcodec                               0.10.0+cu128
torchdata                                0.11.0
torchinfo                                1.8.0
torchmetrics                             1.9.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.25.0+cu128
tqdm                                     4.67.3


In [4]:
#!pip install -r requirements.txt

# Example - ( MNIST Model with CNN )

In [5]:
%%writefile dataset-mnist.py

import argparse
import torchvision.datasets as datasets
import torchvision.transforms as transforms

parser = argparse.ArgumentParser()
parser.add_argument("--data-dir", type=str, default="../../datasets/data")
args = parser.parse_args()

print(f"[*] Downloading MNIST to {args.data_dir} ...")
datasets.MNIST(args.data_dir, train=True,  download=True, transform=transforms.ToTensor())
datasets.MNIST(args.data_dir, train=False, download=True, transform=transforms.ToTensor())
print("[✓] MNIST download complete.")

Writing dataset-mnist.py


In [6]:
!python dataset-mnist.py

[*] Downloading CIFAR-10 to ../../datasets/data ...
100%|██████████████████████████████████████| 9.91M/9.91M [00:01<00:00, 9.61MB/s]
100%|███████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 338kB/s]
100%|██████████████████████████████████████| 1.65M/1.65M [00:00<00:00, 3.19MB/s]
100%|██████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 21.6MB/s]
[✓] CIFAR-10 download complete.


In [9]:
%%writefile mnist-code.py

import time 
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class MnistModel(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.block_1 = nn.Sequential(
            nn.Conv2d(input_shape, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_units * 7 * 7, output_shape),
        )

    def forward(self, x: torch.Tensor):
        x = self.block_1(x)
        x = self.block_2(x)
        x = self.classifier(x)
        return x

def train(model, train_loader, loss_fn, optimizer, epoch, device):

    start_time = time.time()

    model.train()
    total_loss = 0
    correct = 0
    total_samples = 0

    for x_train, y_train in train_loader:
        x_train, y_train = x_train.to(device), y_train.to(device)

        y_pred = model(x_train)  # Forward pass
        loss = loss_fn(y_pred, y_train)  # Compute loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predicted_labels = y_pred.argmax(dim=1)
        correct += (predicted_labels == y_train).sum().item()
        total_samples += y_train.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total_samples

    epoch_time = time.time() - start_time

    print(f"Epoch [{epoch}] | Train-Loss: {avg_loss:.4f} | Acc: {accuracy:.2f}% | Time: {epoch_time:.2f}s")


def test(model, test_loader, loss_fn, epoch, device):

    start_time = time.time()

    model.eval()
    total_loss = 0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(device), y_test.to(device)

            y_pred = model(x_test)
            loss = loss_fn(y_pred, y_test)
            total_loss += loss.item()

            predicted_labels = y_pred.argmax(dim=1)
            correct += (predicted_labels == y_test).sum().item()
            total_samples += y_test.size(0)

    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / total_samples

    epoch_time = time.time() - start_time

    print(f"Epoch [{epoch}] | Test Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}% | Time: {epoch_time:.2f}s")

def main():

    batch_size = 64
    epochs = 3

    torch.manual_seed(42)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    train_dataset = datasets.MNIST("../../datasets/data", train=True, download=False, transform=transform)
    test_dataset = datasets.MNIST("../../datasets/data", train=False, download=False, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, pin_memory=True)

    model = MnistModel(input_shape=1, hidden_units=10, output_shape=10).to(device)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(1, epochs + 1):
        train(model, train_loader, loss_fn, optimizer, epoch, device)
        test(model, test_loader, loss_fn, epoch, device)

if __name__ == "__main__":
    main()


Overwriting mnist-code.py


In [10]:
! python mnist-code.py

Epoch [1] | Train-Loss: 0.1479 | Acc: 95.39% | Time: 14.97s
Epoch [1] | Test Loss: 0.1075 | Accuracy: 96.59% | Time: 2.03s
Epoch [2] | Train-Loss: 0.0766 | Acc: 97.66% | Time: 14.11s
Epoch [2] | Test Loss: 0.0835 | Accuracy: 97.34% | Time: 2.06s
Epoch [3] | Train-Loss: 0.0696 | Acc: 97.89% | Time: 13.94s
Epoch [3] | Test Loss: 0.0711 | Accuracy: 97.81% | Time: 1.99s
Epoch [4] | Train-Loss: 0.0699 | Acc: 97.92% | Time: 13.90s
Epoch [4] | Test Loss: 0.0759 | Accuracy: 98.04% | Time: 1.95s
Epoch [5] | Train-Loss: 0.0705 | Acc: 97.83% | Time: 13.77s
Epoch [5] | Test Loss: 0.0567 | Accuracy: 98.31% | Time: 1.97s
^C
Traceback (most recent call last):
  File "/kaggle/working/mnist-code.py", line 124, in <module>
    main()
  File "/kaggle/working/mnist-code.py", line 120, in main
    train(model, train_loader, loss_fn, optimizer, epoch, device)
  File "/kaggle/working/mnist-code.py", line 48, in train
    for x_train, y_train in train_loader:
                            ^^^^^^^^^^^^
  File "/